<a href="https://colab.research.google.com/github/BOBBYKIRAN/ai-mentor-portfolio/blob/main/Day9_StudyAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langgraph langchain-google-genai langchain-community duckduckgo-search


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [4]:
!
!pip install -U ddgs
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information.
    Use when the question requires current events, recent facts, or
    information not in static training knowledge."""
    return DuckDuckGoSearchRun().run(query)

# Test the tool directly
print(web_search.invoke({'query': 'TCS hiring 2026'})[:400])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 6.7 MB/s eta 0:00:00
February 25, 2026 - TCS will conduct the hiring round using the TCS iON platform rather than the TCS Community platform, which TCS has utilized in the past. Yes, students from 2024 can apply in this drive. January 20, 2026 - Tata Consultancy Services (TCS), India’s most trusted IT services company, has launched its TCS Hiring 2026 drive, creating a massive wave of job opportunities for freshers an


In [5]:
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
agent = create_react_agent(llm, tools=[web_search])

print('Agent created.')

Agent created.


/tmp/ipykernel_826/1070485237.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=[web_search])


In [6]:
result = agent.invoke({
    'messages': [('user', "What is TCS's 2026 hiring quota?")]
})

# Print every message in the conversation
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    Content: {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        print(f'    Tool calls: {m.tool_calls}')


[0] HumanMessage
    Content: What is TCS's 2026 hiring quota?

[1] AIMessage
    Content: 
    Tool calls: [{'name': 'web_search', 'args': {'query': 'TCS 2026 hiring quota'}, 'id': '352128db-4e9b-42c7-89b4-4fafba0468bf', 'type': 'tool_call'}]

[2] ToolMessage
    Content: 1 week ago - TCS NQT 2026 Recruitment is open for 2024–2026 batches. Check eligibility, salary up to ₹12.26 LPA, exam pattern, important dates, and how to apply online. 1 week ago - For 2026, TCS has retained the 60% aggregate threshold introduced in 2022 but tightened its standing arrears policy, c

[3] AIMessage
    Content: [{'type': 'text', 'text': 'I could not find a specific hiring quota for TCS for 2026 in my search results. However, TCS has announced the NQT (National Qualifier Test) hiring drive for the 2026 batch across India, targeting candidates from the 2024, 2025, and 2026 batches for Prime and Digital roles


In [7]:
# Pass a question that should fail the tool
result = agent.invoke({
    'messages': [('user', 'Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd')]
})

# Watch how the agent recovers
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    {str(m.content)[:300]}')


[0] HumanMessage
    Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd

[1] AIMessage
    

[2] ToolMessage
    It will give you the status of whether the domain name that you are looking for is available or not. Doing this for multiple domain names over time can help you build a list of available domain names. Use our WHOIS lookup tool to check domain name availability or to discover the contact information 

[3] AIMessage
    [{'type': 'text', 'text': 'The domain "this-domain-does-not-exist-12345.example.com" likely does not exist. The search results provided information on how to check if a domain name is available, WHOIS lookup tools, and how to find alternative domain names. It also mentioned that if you query a domai


In [8]:
import requests
from bs4 import BeautifulSoup
from langchain_core.tools import tool

@tool
def jd_fetcher(url: str) -> str:
    """Fetch a job description from a URL and return clean plain text.
    Use when the user provides a job posting URL and you need the JD content.
    Returns first 4000 characters of the cleaned page text."""
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        for tag in soup(['script', 'style']):
            tag.decompose()
        return soup.get_text(separator='\n', strip=True)[:4000]
    except Exception as e:
        return f'ERROR: failed to fetch URL — {e}'

In [9]:
@tool
def skills_gap(student_skills: str, must_have_skills: str) -> str:
    """Compare a student's skills (comma-separated) to a job's must-have skills (comma-separated).
    Returns missing skills, comma-separated, or 'none' if student has all.
    Use when the user provides a student profile and a JD's required skills."""
    a = set(s.strip().lower() for s in student_skills.split(',') if s.strip())
    b = set(s.strip().lower() for s in must_have_skills.split(',') if s.strip())
    missing = sorted(b - a)
    return ', '.join(missing) if missing else 'none'

# Test
print(skills_gap.invoke({
    'student_skills': 'Python, Java, SQL',
    'must_have_skills': 'Python, Java, SQL, Spring Boot, AWS',
}))
# Expected: 'aws, spring boot'

aws, spring boot


In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

@tool
def answer_scorer(question: str, answer: str) -> str:
    """Score a student's answer to a placement interview question, 1-10, with one-line rationale.
    Use when evaluating how well a student answered a specific interview question.
    Returns format: 'Score: X/10. Rationale: <reason>'."""
    prompt = (f'Score this placement interview answer 1-10 with one-line rationale.\n'
              f'Question: {question}\n'
              f'Answer: {answer}')
    return llm.invoke(prompt).content

# Test
print(answer_scorer.invoke({
    'question': 'Why TCS Digital?',
    'answer': 'Because TCS is big and they pay well.',
}))
# Expected: low score (~3-4/10), rationale about lack of specificity / cultural fit

**Score: 2/10**

**Rationale:** The answer is purely transactional and generic, showing no genuine interest in TCS or its Digital initiatives beyond compensation.


In [11]:
from langgraph.prebuilt import create_react_agent

tools = [jd_fetcher, skills_gap, answer_scorer]
agent = create_react_agent(llm, tools=tools)
print(f'Agent created with {len(tools)} tools.')

Agent created with 3 tools.


/tmp/ipykernel_826/3943891205.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=tools)


In [13]:
import json, pathlib
profiles = json.loads(pathlib.Path('/content/sample_student_resumes_with_target_company.json').read_text())

for i, p in enumerate(profiles):
    print(f'\n{"="*70}')
    target_company = p.get('target_company', 'Unknown Company') # Safely get target_company
    print(f'Student {i+1}: {p["name"]} — {p["branch"]} CGPA {p["cgpa"]} → {target_company}')
    print(f'{"="*70}')

    msg = (f"I am {p['name']}, B.Tech {p['branch']} CGPA {p['cgpa']}, "
           f"skills: {', '.join(p['skills'])}. Target: {target_company}. " # Use the safely retrieved target_company
           f"Plan 3 mock interview questions for me, score one of my sample answers, "
           f"and tell me what skills I need to add to be a strong fit.")

    result = agent.invoke({'messages': [('user', msg)]}, config={'recursion_limit': 10})

    for j, m in enumerate(result['messages']):
        print(f'\n  [{j}] {type(m).__name__}')
        if hasattr(m, 'content') and m.content:
            print(f'      {str(m.content)[:300]}')
        if hasattr(m, 'tool_calls') and m.tool_calls:
            for tc in m.tool_calls:
                print(f'      → tool_call: {tc.get("name")}({tc.get("args")})')


Student 1: Rahul Verma — CSE CGPA 8.4 → TCS

  [0] HumanMessage
      I am Rahul Verma, B.Tech CSE CGPA 8.4, skills: Python, Java, SQL, Spring Boot, AWS. Target: TCS. Plan 3 mock interview questions for me, score one of my sample answers, and tell me what skills I need to add to be a strong fit.

  [1] AIMessage
      [{'type': 'text', 'text': "Hello Rahul! Based on your profile (B.Tech CSE, skills: Python, Java, SQL, Spring Boot, AWS), here are 3 mock interview questions for TCS:\n\n1.  Explain the difference between a `LinkedList` and an `ArrayList` in Java. When would you prefer one over the other?\n2.  Descri

Student 2: Sneha Reddy — CSE CGPA 8.9 → Infosys

  [0] HumanMessage
      I am Sneha Reddy, B.Tech CSE CGPA 8.9, skills: Python, Java, SQL, Spring Boot, AWS. Target: Infosys. Plan 3 mock interview questions for me, score one of my sample answers, and tell me what skills I need to add to be a strong fit.

  [1] AIMessage
      [{'type': 'text', 'text': "Great to hear about yo

In [14]:
# Pass a bad URL — see how agent recovers
result = agent.invoke({
    'messages': [('user', 'Fetch this JD and tell me the must-have skills: '
                          'https://this-does-not-exist-99999.example.com/jd')]
}, config={'recursion_limit': 5})

print('Failure recovery trace:')
for j, m in enumerate(result['messages']):
    print(f'\n[{j}] {type(m).__name__}')
    if hasattr(m, 'content') and m.content:
        print(f'    {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        for tc in m.tool_calls:
            print(f'    → {tc.get("name")}({tc.get("args")})')

Failure recovery trace:

[0] HumanMessage
    Fetch this JD and tell me the must-have skills: https://this-does-not-exist-99999.example.com/jd

[1] AIMessage
    → jd_fetcher({'url': 'https://this-does-not-exist-99999.example.com/jd'})

[2] ToolMessage
    ERROR: failed to fetch URL — HTTPSConnectionPool(host='this-does-not-exist-99999.example.com', port=443): Max retries exceeded with url: /jd (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7bb845cef7d0>: Failed to resolve 'this-does-not-exist-99999.example.com' ([Errn

[3] AIMessage
    [{'type': 'text', 'text': 'I was unable to fetch the job description from the provided URL. It appears the URL is incorrect or does not exist. Please double-check the URL and try again.', 'extras': {'signature': 'Cp0HAQw51sdjvwRpvrascdLdjSBSwkWU1QA2i1LkAA0/gv3UPw0u6/IRcK4i4VXLYCbqrGdjm6Lq/ZAAmhqrf+L
